In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC



df = pd.read_csv("student_performance.csv")
feature_cols = [
    'study_hours',
    'attendance',
    'internal_score',
    'prev_result',
    'sleep_hours',
    'extra_activity'
]

print("\nDataset Columns:")
print(df.columns)


In [5]:

X = df[feature_cols]


y_score = df['final_score']


y_passfail = (df['final_score'] >= 60).astype(int)



print("\nClass Distribution:")
print(y_passfail.value_counts())


Class Distribution:
final_score
1    481
0     19
Name: count, dtype: int64


In [6]:
X_train, X_test, y_score_train, y_score_test, y_pf_train, y_pf_test = train_test_split(
    X,
    y_score,
    y_passfail,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


In [7]:
lr_model = LinearRegression()

lr_model.fit(
    X_train_scaled,
    y_score_train
)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [8]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_reg.fit(
    X_train_scaled,
    y_score_train
)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
log_model = LogisticRegression()

log_model.fit(
    X_train_scaled,
    y_pf_train
)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [10]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_clf.fit(
    X_train_scaled,
    y_pf_train
)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
svm_model = SVC(
    probability=True
)

svm_model.fit(
    X_train_scaled,
    y_pf_train
)




,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [12]:
def get_float_input(prompt, min_val, max_val):

    while True:

        try:
            value = float(input(prompt))

            if min_val <= value <= max_val:
                return value

            else:
                print(f"Please enter value between {min_val} and {max_val}")

        except ValueError:
            print("Invalid input. Enter numeric value.")


def get_int_input(prompt, min_val, max_val):

    while True:

        try:
            value = int(input(prompt))

            if min_val <= value <= max_val:
                return value

            else:
                print(f"Please enter value between {min_val} and {max_val}")

        except ValueError:
            print("Invalid input. Enter integer value.")

user_study_hours = get_float_input(
    "Study Hours per day (1-10): ",
    1,
    10
)

user_attendance = get_float_input(
    "Attendance Percentage (50-100): ",
    50,
    100
)

user_internal_score = get_float_input(
    "Internal Score (30-100): ",
    30,
    100
)

user_prev_result = get_float_input(
    "Previous Result (30-100): ",
    30,
    100
)

user_sleep_hours = get_float_input(
    "Sleep Hours (4-10): ",
    4,
    10
)

user_extra_activity = get_int_input(
    "Extra Activities (0-3): ",
    0,
    3
)


user_input_raw = np.array([[
    user_study_hours,
    user_attendance,
    user_internal_score,
    user_prev_result,
    user_sleep_hours,
    user_extra_activity
]])



Study Hours per day (1-10):  6
Attendance Percentage (50-100):  78
Internal Score (30-100):  67
Previous Result (30-100):  78
Sleep Hours (4-10):  6
Extra Activities (0-3):  2


In [18]:
user_input_scaled = scaler.transform(user_input_raw)


lr_score_pred = lr_model.predict(user_input_scaled)[0]

rf_score_pred = rf_reg.predict(user_input_scaled)[0]


avg_score_pred = (
    lr_score_pred +
    rf_score_pred
) / 2


avg_score_pred = np.clip(
    avg_score_pred,
    0,
    100
)

C:\Users\utkar\OneDrive\Desktop\New folder\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [21]:
log_pf = log_model.predict(user_input_scaled)[0]

rf_pf = rf_clf.predict(user_input_scaled)[0]

svm_pf = svm_model.predict(user_input_scaled)[0]



log_fail_prob = log_model.predict_proba(
    user_input_scaled
)[0][0]

rf_fail_prob = rf_clf.predict_proba(
    user_input_scaled
)[0][0]

svm_fail_prob = svm_model.predict_proba(
    user_input_scaled
)[0][0]


In [22]:
avg_fail_prob = (
    log_fail_prob +
    rf_fail_prob +
    svm_fail_prob
) / 3



votes = [log_pf, rf_pf, svm_pf]

final_pf = 1 if sum(votes) >= 2 else 0

pf_label = "PASS" if final_pf == 1 else "FAIL"
if avg_fail_prob >= 0.70:

    risk = "HIGH RISK"

elif avg_fail_prob >= 0.40:

    risk = "MEDIUM RISK"

else:

    risk = "LOW RISK"

In [23]:
print("                    INPUT SUMMARY")
print("============================================================")

print(f"Study Hours       : {user_study_hours}")

print(f"Attendance        : {user_attendance}")

print(f"Internal Score    : {user_internal_score}")

print(f"Previous Result   : {user_prev_result}")

print(f"Sleep Hours       : {user_sleep_hours}")

print(f"Extra Activities  : {user_extra_activity}")



print("\n============================================================")
print("                 PREDICTION RESULTS")
print("============================================================")

print(f"Predicted Score   : {avg_score_pred:.2f}")

print(f"Prediction Result : {pf_label}")

print(f"Fail Probability  : {avg_fail_prob:.2%}")

print(f"Risk Level        : {risk}")

print("============================================================")

                    INPUT SUMMARY
Study Hours       : 6.0
Attendance        : 78.0
Internal Score    : 67.0
Previous Result   : 78.0
Sleep Hours       : 6.0
Extra Activities  : 2

                 PREDICTION RESULTS
Predicted Score   : 87.00
Prediction Result : PASS
Fail Probability  : 0.27%
Risk Level        : LOW RISK


In [26]:

print("\n PERSONALIZED RECOMMENDATIONS:")

recommendations = []


if user_study_hours < 3:
    recommendations.append(
        "Increase study time significantly. Try studying at least 5-6 hours daily."
    )
elif user_study_hours < 5:
    recommendations.append(
        "Your study hours are moderate. Increasing to 6 hours may improve scores."
    )
else:
    recommendations.append(
        "Good study habits detected. Maintain consistent study hours."
    )


if user_attendance < 65:
    recommendations.append(
        "Very low attendance detected. Attend classes regularly to avoid academic risk."
    )
elif user_attendance < 75:
    recommendations.append(
        "Improve attendance to above 75% for better academic performance."
    )
else:
    recommendations.append(
        "Attendance level is good. Keep maintaining regular attendance."
    )


if user_internal_score < 50:
    recommendations.append(
        "Internal assessment performance is weak. Focus on assignments and class tests."
    )
elif user_internal_score < 70:
    recommendations.append(
        "Internal scores are average. More revision and practice can help improve marks."
    )
else:
    recommendations.append(
        "Strong internal performance. Continue your current preparation strategy."
    )


if user_prev_result < 50:
    recommendations.append(
        "Previous academic performance is low. Revise fundamentals from earlier subjects."
    )
elif user_prev_result < 70:
    recommendations.append(
        "Previous results are average. Practice weak subjects consistently."
    )
else:
    recommendations.append(
        "Previous academic performance is strong."
    )


if user_sleep_hours < 5:
    recommendations.append(
        "Very low sleep detected. Lack of sleep may reduce concentration and memory."
    )
elif user_sleep_hours < 6:
    recommendations.append(
        "Try getting at least 6-8 hours of sleep daily for better focus."
    )
else:
    recommendations.append(
        "Healthy sleep schedule detected."
    )


if user_extra_activity == 0:
    recommendations.append(
        "Consider participating in extracurricular activities for overall development."
    )
elif user_extra_activity > 2:
    recommendations.append(
        "Too many extracurricular activities may affect study time. Maintain balance."
    )
else:
    recommendations.append(
        "Good balance between academics and extracurricular activities."
    )

if avg_fail_prob >= 0.70:
    recommendations.append(
        "High academic risk detected. Immediate mentoring and additional coaching recommended."
    )
elif avg_fail_prob >= 0.40:
    recommendations.append(
        "Moderate academic risk detected. Regular monitoring and revision are advised."
    )
else:
    recommendations.append(
        "Low academic risk. Continue maintaining your performance."
    )


if avg_score_pred >= 85:
    recommendations.append(
        "Excellent predicted performance! Aim for distinction and advanced learning."
    )
elif avg_score_pred >= 70:
    recommendations.append(
        "Good predicted score. More practice can help achieve excellent results."
    )
elif avg_score_pred >= 50:
    recommendations.append(
        "Average predicted score. Focus on weak subjects and revision."
    )
else:
    recommendations.append(
        "Predicted score is low. A structured study plan is strongly recommended."
    )


for i, rec in enumerate(recommendations, start=1):
    print(f"   {i}. {rec}")



 PERSONALIZED RECOMMENDATIONS:
   1. Good study habits detected. Maintain consistent study hours.
   2. Attendance level is good. Keep maintaining regular attendance.
   3. Internal scores are average. More revision and practice can help improve marks.
   4. Previous academic performance is strong.
   5. Healthy sleep schedule detected.
   6. Good balance between academics and extracurricular activities.
   7. Low academic risk. Continue maintaining your performance.
   8. Excellent predicted performance! Aim for distinction and advanced learning.
